## Classification Model Development

In [1]:
#Set the path to the dataset. 
data_dir = 'C:/Users/Asus/Documents/trendcatcher/data'   

###  1. Setup and Dataset Loading

In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image

In [3]:
# Check if the data directory exists
if not os.path.exists(data_dir):
    print(f"Error: Data directory '{data_dir}' not found. Please download and organize the dataset.")
    exit()

In [4]:
# Define data transformations
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
# Load the dataset
trainset = torchvision.datasets.ImageFolder(root=os.path.join(data_dir, 'train'), transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True, num_workers=2)

testset = torchvision.datasets.ImageFolder(root=os.path.join(data_dir, 'valid'), transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False, num_workers=2)

classes = trainset.classes

### 2. Load Pre-trained EfficientNet-b0 (Feature Extraction)

Load the pre-trained EfficientNet-b0 model

In [6]:
model_feature_extraction = torchvision.models.efficientnet_b0(pretrained=True)

c:\Users\Asus\miniconda3\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Asus\miniconda3\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
# Freeze all layers (feature extraction)
for param in model_feature_extraction.parameters():
    param.requires_grad = False

# Modify the classifier
num_features = model_feature_extraction.classifier[1].in_features
model_feature_extraction.classifier[1] = nn.Linear(num_features, len(classes))

# Move the model to the device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_feature_extraction.to(device)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

### 3. Train the Feature Extraction Model

In [8]:
# Setting the hyperparameters
criterion = nn.CrossEntropyLoss()
optimizer_feature_extraction = optim.Adam(model_feature_extraction.classifier[1].parameters(), lr=0.001)

In [9]:
num_epochs = 5
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)
        optimizer_feature_extraction.zero_grad()
        outputs = model_feature_extraction(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_feature_extraction.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Feature Extraction Loss: {running_loss/len(trainloader)}")

Epoch 1, Feature Extraction Loss: 1.394343666235606
Epoch 2, Feature Extraction Loss: 1.0756545424461366
Epoch 3, Feature Extraction Loss: 0.8900856872399648
Epoch 4, Feature Extraction Loss: 0.816276756922404
Epoch 5, Feature Extraction Loss: 0.7493010401725769


### 4. Evaluation of Feature Extraction Model

In [10]:
correct = 0
total = 0
y_true_fe = []
y_pred_fe = []

with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model_feature_extraction(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        y_true_fe.extend(labels.cpu().numpy())
        y_pred_fe.extend(predicted.cpu().numpy())

print(f"Accuracy of Feature Extraction model: {100 * correct / total}%")
print(classification_report(y_true_fe, y_pred_fe, target_names=classes))

Accuracy of Feature Extraction model: 34.48275862068966%
              precision    recall  f1-score   support

  athleisure       0.36      0.08      0.13       103
   fairycore       0.16      0.22      0.18        60
   old money       0.48      0.52      0.50       262
  streetwear       0.30      0.34      0.32       133
         y2k       0.10      0.14      0.12        51

    accuracy                           0.34       609
   macro avg       0.28      0.26      0.25       609
weighted avg       0.36      0.34      0.33       609



### 5. Load Pre-trained EfficientNet-b0 (Fine-tuning)

In [11]:
model_fine_tuning = torchvision.models.efficientnet_b0(pretrained=True)
num_features = model_fine_tuning.classifier[1].in_features
model_fine_tuning.classifier[1] = nn.Linear(num_features, len(classes))
model_fine_tuning.to(device)

c:\Users\Asus\miniconda3\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Asus\miniconda3\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

### 6. Train the Fine-tuning Model

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer_fine_tuning = optim.Adam(model_fine_tuning.parameters(), lr=0.0001) # Lower learning rate for fine-tuning

In [13]:
# Adjust the num_epochs as needed. This cell can take several minutes with CPU.

num_epochs = 5
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)
        optimizer_fine_tuning.zero_grad()
        outputs = model_fine_tuning(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_fine_tuning.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Fine-tuning Loss: {running_loss/len(trainloader)}")

Epoch 1, Fine-tuning Loss: 1.4368862827618918
Epoch 2, Fine-tuning Loss: 1.0546553969383239
Epoch 3, Fine-tuning Loss: 0.7312570015589396
Epoch 4, Fine-tuning Loss: 0.5773495336373647
Epoch 5, Fine-tuning Loss: 0.43203023870786034


### 7. Evaluation of Fine-tuning Model

In [14]:
correct = 0
total = 0
y_true_ft = []
y_pred_ft = []

with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model_fine_tuning(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        y_true_ft.extend(labels.cpu().numpy())
        y_pred_ft.extend(predicted.cpu().numpy())

print(f"Accuracy of Fine-tuning model: {100 * correct / total}%")
print(classification_report(y_true_ft, y_pred_ft, target_names=classes))

Accuracy of Fine-tuning model: 33.66174055829228%
              precision    recall  f1-score   support

  athleisure       0.37      0.11      0.17       103
   fairycore       0.14      0.22      0.17        60
   old money       0.49      0.53      0.51       262
  streetwear       0.28      0.26      0.27       133
         y2k       0.10      0.16      0.12        51

    accuracy                           0.34       609
   macro avg       0.28      0.25      0.25       609
weighted avg       0.36      0.34      0.33       609



### 8. Comparison of Models

In [15]:
print("Comparison of Models:")
print("Feature Extraction Model:")
print(classification_report(y_true_fe, y_pred_fe, target_names=classes))
print("Fine-tuning Model:")
print(classification_report(y_true_ft, y_pred_ft, target_names=classes))

Comparison of Models:
Feature Extraction Model:
              precision    recall  f1-score   support

  athleisure       0.36      0.08      0.13       103
   fairycore       0.16      0.22      0.18        60
   old money       0.48      0.52      0.50       262
  streetwear       0.30      0.34      0.32       133
         y2k       0.10      0.14      0.12        51

    accuracy                           0.34       609
   macro avg       0.28      0.26      0.25       609
weighted avg       0.36      0.34      0.33       609

Fine-tuning Model:
              precision    recall  f1-score   support

  athleisure       0.37      0.11      0.17       103
   fairycore       0.14      0.22      0.17        60
   old money       0.49      0.53      0.51       262
  streetwear       0.28      0.26      0.27       133
         y2k       0.10      0.16      0.12        51

    accuracy                           0.34       609
   macro avg       0.28      0.25      0.25       609
weighted a

### 9. Save the Models

In [16]:
torch.save(model_feature_extraction.state_dict(), 'styles_feature_extraction.pth')
torch.save(model_fine_tuning.state_dict(), 'styles_fine_tuning.pth')

## Deploy Your Best Image Classification Model with Streamlit

In [ ]:
#run on streamlit
!streamlit run app.py

^C


<p style="text-align:center;">That's it! Congratulations! <br> 
    Now, call an LA to check your solution. Then, upload your code on MyCourses.</p>